In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/10 09:32:42 WARN Utils: Your hostname, MARYs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.132.115 instead (on interface en0)
26/03/10 09:32:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/10 09:32:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
spark.version

'4.1.1'

In [4]:
df = spark.read \
    .option("header", "true") \
    .parquet('yellow_tripdata_2025-11.parquet')

In [5]:
df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [6]:
df.schema

StructType([StructField('VendorID', IntegerType(), True), StructField('tpep_pickup_datetime', TimestampNTZType(), True), StructField('tpep_dropoff_datetime', TimestampNTZType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('Airport_fee', DoubleType(), True), StructField('cbd_congestio

In [7]:
df2 = df.repartition(4)

In [8]:
output_path = '/Users/marychou/Documents/dezoomcamp/github/de-zoomcamp-homework/module6/repartition'

In [10]:
df2.write.mode("overwrite").parquet(output_path)

In [11]:
df2.schema

StructType([StructField('VendorID', IntegerType(), True), StructField('tpep_pickup_datetime', TimestampNTZType(), True), StructField('tpep_dropoff_datetime', TimestampNTZType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('Airport_fee', DoubleType(), True), StructField('cbd_congestio

In [12]:
df2.createOrReplaceTempView("temp_view")

In [18]:
query = """
    SELECT COUNT(*) as count
    FROM temp_view
    WHERE tpep_pickup_datetime >= TIMESTAMP('2025-11-15 00:00:00')
    AND tpep_pickup_datetime < TIMESTAMP('2025-11-16 00:00:00')
    """

In [19]:
sql_result_df = spark.sql(query)

In [20]:
sql_result_df.show()

+------+
| count|
+------+
|162604|
+------+



In [21]:
query_longest = """
    SELECT MAX(months_between(tpep_dropoff_datetime, tpep_pickup_datetime)) AS maxtime_trip FROM temp_view
    """

In [22]:
sql_result_df = spark.sql(query_longest)

In [23]:
sql_result_df.show()

+------------+
|maxtime_trip|
+------------+
|  0.12183692|
+------------+



In [24]:
0.12183692*31*24

90.64666848

In [99]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-10 10:21:01--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.161.140.125, 18.161.140.201, 18.161.140.41, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.161.140.125|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv.1’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-10 10:21:02 (905 MB/s) - ‘taxi_zone_lookup.csv.1’ saved [12331/12331]



In [101]:
df = spark.read \
    .option("header", "true") \
    .csv('taxi_zone_lookup.csv')

In [102]:
df_zones.schema

StructType([StructField('LocationID', StringType(), True), StructField('Borough', StringType(), True), StructField('Zone', StringType(), True), StructField('service_zone', StringType(), True)])

In [103]:
df_zones.createOrReplaceTempView("zones_view")

In [104]:
query_fewest = """
    SELECT Z.* FROM zones_view Z
    INNER JOIN
    (
    SELECT PULocationID, COUNT(*) AS cnt
    FROM temp_view
    GROUP BY PULocationID
    HAVING COUNT(*) =
    (
    SELECT MIN(cnt) min_cnt FROM
    (
    SELECT PULocationID, COUNT(*) AS cnt
    FROM temp_view
    GROUP BY PULocationID
    )
    )
    )
    ON PULocationID = Z.LocationID
    """

In [105]:
sql_result_df3 = spark.sql(query_fewest)

In [106]:
sql_result_df3.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|        84|Staten Island|Eltingville/Annad...|   Boro Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|       105|    Manhattan|Governor's Island...| Yellow Zone|
+----------+-------------+--------------------+------------+

